# 문서를 불러오고 잘게 쪼개기

**위니브 카페 CS 챗봇 만들기**


## 오늘 배울 것 

> 문서를 **통째로** 넣었더니 길면 토큰비용이 많이 듭니다..
> 그래서 문서를 **잘게 쪼개** 두는 법을 배웁니다. 

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 왜 불러오고 왜 쪼개나? |
| 2️⃣ 기본 | 파일 불러오기 + 쪼개기 (가장 쉬운 형태) |
| 3️⃣ 응용 | 쪼개는 크기를 바꿔보며 차이 확인 |
| 4️⃣ 실습 | 위니브 문서 5개를 쪼개고 분류 꼬리표 달기 |


# 1️⃣ 단계 — 이론 

## 숙제

위니브 규정을 **통째로** 넣었더니 잘 답했죠. 하지만
마지막 셀에서 봤듯이, 문서가 길어지면 AI가 읽을 양(토큰)이
폭증해서 **비싸고 느려집니다.** 위니브 문서가 5개나 되는데
다 통째로 넣을 순 없습니다.

> 🗝️ 그래서 결론: **"통째로 말고, 질문에 필요한 부분만 주자."**

그런데 "필요한 부분만" 주려면, 먼저 두 가지 준비가 필요합니다.

## 준비 ① 문서를 "불러온다"

위니브 문서는 지금 그냥 **파일**(.md)입니다. 사람은 더블클릭해서
읽지만, 우리 챗봇 프로그램은 파일을 **자기 안으로 읽어 들여야**
다룰 수 있습니다. 이 일을 하는 도구가 **Document Loader** 입니다.

> 💡 비유: 종이 서류를 컴퓨터에서 쓰려면 먼저 **스캔**해야 하죠.
> Document Loader가 그 스캐너입니다.

불러온 결과에는 두 가지가 들어 있습니다:
- **본문** : 실제 글 내용
- **꼬리표** : 이 글이 어느 파일에서 왔는지 (출처 정보)

> 🔑 이 꼬리표는 나중에 아주 중요합니다. 4교시에서 "출처: 환불정책"
> 처럼 답에 근거를 붙일 때, 5교시에서 "원두 문서만 검색"할 때 씁니다.

## 준비 ② 문서를 "쪼갠다"

불러온 문서를 그대로 안 쓰고 **잘게 자릅니다.** 왜 멀쩡한 문서를
토막 낼까요?

1. **검색을 잘 하려면 작아야 함**: 다음 시간에 "질문에 맞는 부분만"
   찾을 건데, 검색 단위가 문서 전체면 1교시 문제로 돌아갑니다.
   조각으로 잘라야 "환불 부분"만 콕 집을 수 있습니다.
2. **AI가 한 번에 읽는 양에 한계가 있음**: 작은 조각으로 나눠 두면
   필요한 몇 조각만 골라 쓸 수 있습니다.

> 잘린 한 조각을 **청크(Chunk)** 라고 부릅니다. (덩어리/토막)

## 어떻게 쪼개나 — 두 개의 손잡이

- **chunk_size** : 한 조각을 얼마나 크게 자를지 (글자 수)
  - 너무 작으면: 문맥이 끊겨 의미가 사라짐
  - 너무 크면: 1교시 문제 재발 (비싸고 잡음 많음)
- **chunk_overlap** : 이웃 조각끼리 얼마나 겹칠지
  - 0이면 문장 한가운데가 뚝 잘릴 수 있음 → 조금 겹쳐서 안전하게

> 💡 비유(overlap): 긴 영상을 토막 낼 때 장면이 뚝 끊기지 않게
> 앞뒤를 몇 초씩 겹쳐 자르는 것과 같습니다.

---
👉 아래부터 코드입니다. 한 줄씩 천천히 갑니다.


# 2️⃣ 단계 — 파일 불러오고 쪼개보기

작은 예제로 "불러오기"와 "쪼개기"가 뭔지 가장 쉬운 형태로
확인합니다.

### 셀 1 — 도구 설치

문서를 불러오고(loader) 쪼개는(splitter) 도구를 설치합니다.

In [2]:
!pip install -q langchain langchain-community langchain-text-splitters python-dotenv

### 셀 2 — 연습용 파일 하나 만들기

실무에서 문서는 **파일**로 존재합니다. 먼저 작은 연습 파일을
하나 만들어 봅니다.

In [3]:
# 연습용 텍스트 파일 1개 생성
text = """위니브 카페 안내
영업시간은 오전 10시부터 오후 9시까지입니다.
주차는 1시간 무료로 제공됩니다."""

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("sample.txt 파일을 만들었습니다.")

sample.txt 파일을 만들었습니다.


**이 셀이 한 일**: `sample.txt` 라는 연습용 파일을 만들었습니다.

### 셀 3 — 파일 불러오기 (Document Loader)

방금 만든 파일을 프로그램 안으로 읽어들입니다.
`TextLoader` 가 그 일을 합니다.

In [4]:
from langchain_community.document_loaders import TextLoader

# 파일을 읽어들이기
loader = TextLoader("sample.txt", encoding="utf-8")
docs = loader.load()

# 불러온 결과 확인 (docs는 리스트, 첫 번째 문서를 봄)
doc = docs[0]
print("본문:", doc.page_content)
print()
print("꼬리표(어디서 왔는지):", doc.metadata)

본문: 위니브 카페 안내
영업시간은 오전 10시부터 오후 9시까지입니다.
주차는 1시간 무료로 제공됩니다.

꼬리표(어디서 왔는지): {'source': 'sample.txt'}


**이 셀이 한 일**: 파일을 읽어 **본문 + 꼬리표** 형태로
가져왔습니다.

> 💡 `doc.page_content` = 본문 / `doc.metadata` = 꼬리표(출처).
> 꼬리표에 `{'source': 'sample.txt'}` 처럼 어디서 왔는지가 자동으로
> 붙어 있습니다. — 1단계에서 말한 그 중요한 꼬리표입니다.

### 셀 4 — 문서 쪼개기 (Text Splitter)

긴 글을 작은 조각(청크)으로 자릅니다.
`chunk_size` 와 `chunk_overlap` 만 정하면 됩니다.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 긴 연습 문장
long_text = "첫째 문장입니다. 둘째 문장입니다. 셋째 문장입니다. 넷째 문장입니다. 다섯째 문장입니다."

# 한 조각을 25자 정도로, 이웃과 5자 겹치게 자르기
splitter = RecursiveCharacterTextSplitter(chunk_size=25, chunk_overlap=5)
pieces = splitter.split_text(long_text)

print("조각 개수:", len(pieces))
for i, p in enumerate(pieces):
    print(f"[{i}] {p}")

조각 개수: 3
[0] 첫째 문장입니다. 둘째 문장입니다. 셋째
[1] 셋째 문장입니다. 넷째 문장입니다. 다섯째
[2] 다섯째 문장입니다.


**이 셀이 한 일**: 긴 문장을 작은 조각 여러 개로 잘랐습니다.

> 💡 각 조각이 약 25자이고, 조각 [0]의 끝과 [1]의 시작에 겹치는
> 글자가 보이면 `overlap`(겹침)이 작동한 것입니다.

### 셀 5 — 파일을 불러와서 바로 쪼개기

실전에서는 셀 3(불러오기) + 셀 4(쪼개기)를 이어서 합니다.
`split_documents` 를 쓰면 **꼬리표(출처)가 조각마다 그대로
복사**됩니다 — 이게 핵심입니다.

In [6]:
# 셀 3에서 불러온 docs 를 쪼개기
splitter = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=5)
chunks = splitter.split_documents(docs)

print("조각 개수:", len(chunks))
for c in chunks:
    print("본문:", c.page_content, "| 꼬리표:", c.metadata)

조각 개수: 4
본문: 위니브 카페 안내 | 꼬리표: {'source': 'sample.txt'}
본문: 영업시간은 오전 10시부터 오후 | 꼬리표: {'source': 'sample.txt'}
본문: 오후 9시까지입니다. | 꼬리표: {'source': 'sample.txt'}
본문: 주차는 1시간 무료로 제공됩니다. | 꼬리표: {'source': 'sample.txt'}


**이 셀이 한 일**: 파일을 조각으로 자르면서, 모든 조각에
원본 출처(`sample.txt`) 꼬리표가 그대로 따라왔습니다.

> 📌 정리: **불러오기**(`loader.load()`) → **쪼개기**
> (`splitter.split_documents()`). 이 두 줄이 2교시의 핵심입니다.

# 3️⃣ 단계 — 쪼개는 크기를 바꿔보기

`chunk_size` 를 바꾸면 결과가 어떻게 달라질까요?
"정답 숫자"는 없습니다. **감각**을 기르는 게 목표입니다.

### 셀 6 — 크기를 바꿔가며 조각 수 비교

같은 글을 여러 크기로 잘라, 조각이 몇 개 나오는지 봅니다.

In [7]:
base_text = ("위니브 카페는 스페셜티 원두를 판매합니다. "
             "신선한 로스팅을 자랑합니다. 드립백도 인기입니다. "
             "디저트와 굿즈도 함께 판매합니다.") * 5   # 같은 문장 5번 반복(길게)

for size in [30, 100, 300]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    pieces = sp.split_text(base_text)
    print(f"chunk_size={size:>3} → 조각 {len(pieces)}개")

chunk_size= 30 → 조각 13개
chunk_size=100 → 조각 4개
chunk_size=300 → 조각 2개


**이 셀이 한 일**: 크기를 30 / 100 / 300으로 바꿔 잘랐습니다.

> 💡 크기가 작을수록 조각이 **많아지고**, 클수록 **적어집니다.**
> - 너무 작으면: 문맥이 끊겨 의미 부족
> - 너무 크면: 1교시 문제(비싸고 잡음 많음) 재발
> 적당한 크기를 고르는 감각이 중요합니다.

### 셀 7 — 겹침(overlap) 있을 때 vs 없을 때

`overlap` 이 0이면 문장이 중간에 잘릴 수 있습니다.
0일 때와 있을 때를 비교합니다.

In [8]:
text = "환불은 수령일로부터 7일 이내에 신청해야 하며 개봉 시 불가합니다."

print("───[ overlap=0: 딱 잘림 ]───")
no_overlap = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=0)
for p in no_overlap.split_text(text):
    print(" ", p)

print("\n───[ overlap=8: 이웃과 겹쳐서 안전 ]───")
with_overlap = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=8)
for p in with_overlap.split_text(text):
    print(" ", p)

───[ overlap=0: 딱 잘림 ]───
  환불은 수령일로부터 7일 이내에
  신청해야 하며 개봉 시 불가합니다.

───[ overlap=8: 이웃과 겹쳐서 안전 ]───
  환불은 수령일로부터 7일 이내에
  7일 이내에 신청해야 하며 개봉 시
  하며 개봉 시 불가합니다.


**이 셀이 한 일**: 같은 문장을 겹침 없이 / 있이 잘랐습니다.

> 💡 overlap=0은 조각 경계가 딱 잘려 "7일" 같은 정보가 끊길 수
> 있습니다. overlap=8은 다음 조각이 앞 조각 끝을 다시 포함해서
> 문맥이 안 끊깁니다. — 약간 중복돼도 이게 더 안전합니다.

# 4️⃣ 단계 — 위니브 문서 5개 쪼개기

이제 진짜 위니브 카페 문서 5개를 불러와 청크로 만들고,
나중에 검색에 쓸 **분류 꼬리표**까지 답니다.

### 셀 8 — 위니브 문서 5개를 파일로 저장

강사가 만든 위니브 CS 문서 5개가 아래 변수에 들어 있습니다.
이걸 실제 파일로 저장해 "파일 불러오기"를 진짜로 해봅니다.

In [ ]:
import os

# 위니브 CS 문서 5개 
documents = {
    "01_faq.md":        "# 위니브 카페 자주 묻는 질문 (FAQ)\n\n스페셜티 커피 전문 쇼핑몰 위니브 카페 고객센터입니다.\n자주 묻는 질문을 모았습니다.\n\n---\n\n### Q. 주문은 어떻게 하나요?\nA. 원하시는 상품을 장바구니에 담은 뒤 결제 페이지에서 배송지와 결제수단을\n입력하시면 됩니다. 회원·비회원 모두 주문 가능합니다.\n\n### Q. 비회원도 주문할 수 있나요?\nA. 네, 비회원 주문이 가능합니다. 다만 비회원 주문은 적립금이 적립되지 않으며,\n회원 등급 혜택도 적용되지 않습니다.\n\n### Q. 주문을 취소하고 싶어요.\nA. 상품이 발송되기 전이라면 마이페이지 > 주문내역에서 직접 취소할 수 있습니다.\n이미 발송된 경우에는 반품 절차로 진행됩니다. 자세한 내용은 환불·반품 정책을\n확인해 주세요.\n\n### Q. 결제 수단은 무엇이 있나요?\nA. 신용·체크카드, 계좌이체, 간편결제(카카오페이·네이버페이)를 지원합니다.\n무통장입금은 현재 지원하지 않습니다.\n\n### Q. 선물 포장이 되나요?\nA. 네, 가능합니다. 주문 시 '선물포장' 옵션을 선택하시면 건당 2,000원에\n포장해 드립니다. 선물 메시지 카드도 함께 넣어드립니다.\n\n### Q. 주문한 상품은 언제 받을 수 있나요?\nA. 평일 오후 2시 이전 결제 건은 당일 발송되며, 보통 발송 후 1~2일 이내\n도착합니다. 지역에 따라 더 걸릴 수 있습니다. 배송 정책 문서를 참고해 주세요.\n\n### Q. 배송 조회는 어디서 하나요?\nA. 마이페이지 > 주문내역에서 송장번호와 배송 상태를 확인할 수 있습니다.\n발송 완료 시 알림 문자도 발송됩니다.\n\n### Q. 원두를 분쇄해서 보내주나요?\nA. 네, 주문 시 분쇄도(홀빈·드립·에스프레소 등)를 선택하실 수 있습니다.\n신선도를 위해 홀빈(분쇄 안 함) 구매를 권장드립니다.\n\n### Q. 재고가 없는 상품은 언제 다시 들어오나요?\nA. 품절 상품 페이지에서 '재입고 알림'을 신청하시면 입고 시 문자로\n안내드립니다. 원두는 로스팅 일정에 따라 보통 주 1회 입고됩니다.\n\n### Q. 영수증·세금계산서 발급이 되나요?\nA. 현금영수증은 결제 시 신청 가능하며, 세금계산서는 사업자 회원에 한해\n고객센터로 요청하시면 발급해 드립니다.\n\n### Q. 상품 문의는 어디로 하나요?\nA. 상품 상세페이지의 Q&A 게시판 또는 고객센터 채팅으로 문의해 주세요.\n운영 시간은 평일 오전 10시~오후 5시입니다.",
    "02_refund.md":     "# 위니브 카페 환불·반품·교환 정책\n\n본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.\n\n## 1. 반품·환불 신청 기간\n\n- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능\n- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능\n- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.\n\n## 2. 환불 불가 품목 (중요)\n\n다음 품목은 단순 변심에 의한 반품·환불이 **불가**합니다.\n\n- **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.\n  미개봉 상태에서는 7일 이내 반품 가능합니다.\n- **디저트류(케이크·쿠키·구움과자 등)**: 신선식품으로, 단순 변심 반품이\n  불가합니다. 단, 변질·파손 등 하자가 있는 경우 수령 14일 이내 교환·환불됩니다.\n- **주문 제작 굿즈(각인·커스텀 텀블러 등)**: 고객 요청으로 제작된 상품은\n  단순 변심 반품이 불가합니다.\n- 단, 위 품목도 **상품 하자나 오배송**인 경우에는 환불·교환이 가능합니다.\n\n## 3. 단순 변심 반품 시 비용\n\n- 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원 × 2 = 6,000원)는\n  고객이 부담합니다.\n- 상품 하자·오배송으로 인한 반품은 배송비를 위니브 카페가 전액 부담합니다.\n\n## 4. 환불 처리 절차 및 기간\n\n1. 마이페이지 > 주문내역에서 반품 신청\n2. 반품 상품 발송 (회수 택배 신청 또는 직접 발송)\n3. 반품 상품 도착 후 상태 확인\n4. 확인 완료 후 **영업일 기준 3일 이내** 환불 처리\n\n- 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가로\n  소요될 수 있습니다.\n- 적립금으로 결제한 부분은 적립금으로 환원됩니다.\n\n## 5. 교환\n\n- 동일 상품의 색상·옵션 변경은 재고가 있는 경우 교환 가능합니다.\n- 하자 상품의 교환 배송비는 위니브 카페가 부담합니다.\n- 단순 변심에 의한 교환 배송비는 고객이 부담합니다.\n- 디저트류는 교환이 어려워 환불로 처리될 수 있습니다.\n\n## 6. 주문 취소\n\n- 상품 발송 **전**에는 마이페이지에서 즉시 취소 가능하며 전액 환불됩니다.\n- 발송 **후**에는 취소가 불가하며, 수령 후 반품 절차로 진행됩니다.\n\n## 7. 환불이 제한되는 경우\n\n- 고객의 사용·훼손으로 상품 가치가 현저히 감소한 경우\n- 개봉 후 상태인 식품(원두·드립백·디저트)의 단순 변심\n- 반품 기간(7일/14일)을 경과한 경우\n- 위 사유라도 상품 자체 하자가 입증되면 환불 대상이 됩니다.",
    "03_shipping.md":   "# 위니브 카페 배송 정책\n\n## 1. 배송비\n\n| 구분 | 배송비 |\n|------|--------|\n| 기본 배송비 | 3,000원 |\n| 5만원 이상 구매 | 무료 |\n| 제주 지역 | 기본 배송비 + 3,000원 |\n| 도서산간 지역 | 기본 배송비 + 3,000원 |\n\n- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비\n  (3,000원)는 별도로 부과됩니다.\n\n## 2. 배송 기간\n\n- 평일 **오후 2시 이전** 결제 완료 건은 당일 발송됩니다.\n- 오후 2시 이후 결제 건은 다음 영업일에 발송됩니다.\n- 발송 후 도착까지 보통 1~2일 소요됩니다.\n- 제주·도서산간은 2~4일이 소요될 수 있습니다.\n- 주말·공휴일은 발송되지 않습니다.\n\n## 3. 원두 로스팅 일정\n\n- 원두는 신선도를 위해 **주문 후 로스팅(주문 로스팅)** 으로 발송됩니다.\n- 로스팅은 매주 화·금요일에 진행되며, 해당 요일 직전 주문이 함께\n  발송됩니다. 이로 인해 원두 단일 주문은 최대 3~4일이 소요될 수 있습니다.\n- 원두와 일반 상품을 함께 주문하면 로스팅 일정에 맞춰 합포장 발송됩니다.\n\n## 4. 배송 조회\n\n- 마이페이지 > 주문내역에서 송장번호로 실시간 조회가 가능합니다.\n- 발송 완료 시 등록된 휴대폰으로 알림 문자가 발송됩니다.\n\n## 5. 배송 관련 안내\n\n- 택배사는 일반 택배(CJ대한통운)를 이용합니다.\n- **새벽배송·당일배송 서비스는 운영하지 않습니다.**\n- 부재 시 택배사 정책에 따라 경비실 위탁 또는 재배송됩니다.\n- 배송지 오기입으로 인한 반송 시 재발송 배송비는 고객이 부담합니다.\n\n## 6. 해외 배송\n\n- 현재 해외 배송은 지원하지 않습니다. 국내 배송만 가능합니다.",
    "04_catalog.md":    "# 위니브 카페 상품 정보 카탈로그\n\n각 상품에는 `카테고리` 정보가 있습니다. (메타데이터 필터 실습에 사용)\n카테고리: 원두 / 드립백 / 디저트 / 홈카페기기 / 바리스타굿즈\n\n---\n\n## [원두] 에티오피아 예가체프 G1\n\n- 카테고리: 원두\n- 가격: 18,000원 (250g)\n- 설명: 자스민·베르가못 향이 도는 밝은 산미의 워시드 원두입니다.\n- 보관: 직사광선을 피해 서늘한 곳에 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [원두] 브라질 산토스 NY2\n\n- 카테고리: 원두\n- 가격: 14,000원 (250g)\n- 설명: 견과류·다크초콜릿의 묵직한 바디. 에스프레소 베이스로 인기.\n- 보관: 밀폐 용기 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [드립백] 데일리 블렌드 드립백 10개입\n\n- 카테고리: 드립백\n- 가격: 9,000원 (10개입)\n- 설명: 사무실·여행용 휴대 간편 드립백. 균형 잡힌 블렌드.\n- 보관: 개별 포장, 실온 보관\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [디저트] 클래식 파운드 케이크\n\n- 카테고리: 디저트\n- 가격: 12,000원\n- 설명: 버터 풍미가 진한 수제 파운드 케이크. 커피와 페어링 추천.\n- 보관: 냉장 보관, 수령 후 3일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 변질·파손 시 14일 이내 교환·환불\n\n## [디저트] 비스코티 세트 (6개)\n\n- 카테고리: 디저트\n- 가격: 9,500원\n- 설명: 아몬드·헤이즐넛 두 가지 맛 구움과자 세트.\n- 보관: 실온 보관, 수령 후 7일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 하자 시 14일 이내 교환·환불\n\n## [홈카페기기] 핸드드립 세트 (드리퍼+서버)\n\n- 카테고리: 홈카페기기\n- 가격: 38,000원\n- 설명: 입문용 드리퍼와 600ml 유리 서버 구성.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자 시 14일 이내 교환\n\n## [홈카페기기] 전동 그라인더 GR-200\n\n- 카테고리: 홈카페기기\n- 가격: 89,000원\n- 설명: 40단계 분쇄도 조절 코니컬 버 그라인더.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자(작동 불량) 시 14일 이내 무상 교환\n\n## [바리스타굿즈] 위니브 로고 머그컵\n\n- 카테고리: 바리스타굿즈\n- 가격: 12,000원\n- 설명: 300ml 도자기 머그. 위니브 카페 로고 인쇄.\n- 보관: -\n- 환불: 미사용 7일 이내 가능\n\n## [바리스타굿즈] 커스텀 각인 텀블러 (주문제작)\n\n- 카테고리: 바리스타굿즈\n- 가격: 28,000원\n- 설명: 원하는 문구를 각인해 제작하는 보온 텀블러.\n- 보관: -\n- 환불: **주문 제작 상품으로 단순 변심 반품 불가** / 하자 시 교환 가능",
    "05_membership.md": "# 위니브 카페 멤버십·적립 안내\n\n## 1. 회원 등급\n\n회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일 자동 산정됩니다.\n\n| 등급 | 최근 6개월 누적 구매액 | 적립률 |\n|------|----------------------|--------|\n| 베이직 | 0원 ~ | 1% |\n| 실버 | 10만원 이상 | 2% |\n| 골드 | 30만원 이상 | 3% |\n| VIP | 60만원 이상 | 5% |\n\n- 등급은 매월 1일 재산정되며, 기준 미달 시 하향될 수 있습니다.\n- 신규 가입 시 기본 등급은 베이직입니다.\n\n## 2. 적립금\n\n- 구매 확정 시 등급별 적립률에 따라 적립됩니다.\n- 적립금은 **적립일로부터 1년** 경과 시 자동 소멸됩니다.\n- 적립금은 1,000원 이상부터 사용 가능하며, 1원 단위로 사용할 수 있습니다.\n- 비회원 주문은 적립이 적용되지 않습니다.\n\n## 3. 신규 가입 혜택\n\n- 신규 회원가입 시 3,000원 웰컴 적립금이 즉시 지급됩니다.\n- 웰컴 적립금의 유효기간은 지급일로부터 30일입니다.\n\n## 4. 생일 쿠폰\n\n- 생일이 등록된 회원에게 생일月 1일에 10% 할인 쿠폰이 발급됩니다.\n- 생일 쿠폰은 발급일로부터 그 달 말일까지 사용 가능합니다.\n- 일부 할인 상품에는 중복 적용되지 않습니다.\n\n## 5. 적립금 환불 처리\n\n- 적립금으로 결제한 주문을 환불할 경우, 사용한 적립금은 적립금으로\n  환원됩니다(현금 환불 아님).\n- 환원된 적립금의 유효기간은 원래 적립금의 잔여 기간을 따릅니다.\n\n## 6. 등급 혜택 요약\n\n- 골드 등급 이상: 무료 선물포장 제공\n- VIP 등급: 신상품 우선 구매 및 분기별 시음 원두 증정",
}

# winib_docs 폴더를 만들고 5개 파일 저장
os.makedirs("winib_docs", exist_ok=True)
for filename, content in documents.items():
    with open(f"winib_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("저장한 파일:")
for filename in documents:
    print(" -", filename)

저장한 파일:
 - 01_faq.md
 - 02_refund.md
 - 03_shipping.md
 - 04_catalog.md
 - 05_membership.md


**이 셀이 한 일**: `winib_docs/` 폴더에 위니브 문서 5개를
파일로 저장했습니다.

### 셀 9 — 폴더 전체를 한 번에 불러오기

문서가 5개니 하나씩 부르긴 번거롭죠. 폴더를 통째로 읽습니다.

In [10]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# winib_docs 폴더의 .md 파일을 전부 불러오기
loader = DirectoryLoader(
    "winib_docs",
    glob="*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
all_docs = loader.load()

print("불러온 문서 수:", len(all_docs))
for d in all_docs:
    filename = os.path.basename(d.metadata["source"])
    print(f" - {filename}  ({len(d.page_content)}자)")

불러온 문서 수: 5
 - 02_refund.md  (1264자)
 - 01_faq.md  (1177자)
 - 04_catalog.md  (1437자)
 - 03_shipping.md  (860자)
 - 05_membership.md  (842자)


**이 셀이 한 일**: 폴더의 문서 5개를 한 번에 불러왔습니다.
각 문서의 파일명과 글자 수가 보입니다.

### 셀 10 — 위니브 문서 5개를 청크로 쪼개기

3단계에서 익힌 그대로, 위니브 문서를 청크로 자릅니다.
CS 문서엔 300자 정도가 무난합니다.

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(all_docs)

print("전체 청크 개수:", len(chunks))
print()
print("───[ 첫 번째 청크 ]───")
print("본문:", chunks[0].page_content[:100], "...")
print("꼬리표:", chunks[0].metadata)

전체 청크 개수: 25

───[ 첫 번째 청크 ]───
본문: # 위니브 카페 환불·반품·교환 정책

본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.

## 1. 반품·환불 신청 기간

- **단순 변심**: ...
꼬리표: {'source': 'winib_docs/02_refund.md'}


**이 셀이 한 일**: 문서 5개를 작은 청크 여러 개로 잘랐습니다.

> 💡 청크 개수가 문서 수(5)보다 훨씬 많습니다(잘게 쪼갰으니까).
> 각 청크에 원본 출처 꼬리표가 따라온 것도 확인하세요.

### 셀 11 — 청크에 "분류" 꼬리표 달기

5교시에서 "원두 질문엔 원두 문서만 검색"하려면, 지금 각 청크에
**어떤 분류인지** 꼬리표를 달아둬야 합니다. 파일명으로 분류합니다.

In [12]:
# 파일명 → 분류 이름
category_map = {
    "01_faq.md": "faq",
    "02_refund.md": "refund",
    "03_shipping.md": "shipping",
    "04_catalog.md": "catalog",
    "05_membership.md": "membership",
}

# 각 청크의 꼬리표에 "category" 추가
for c in chunks:
    filename = os.path.basename(c.metadata["source"])
    c.metadata["category"] = category_map[filename]

# 분류별로 청크가 몇 개인지 세기
from collections import Counter
counts = Counter(c.metadata["category"] for c in chunks)
print("분류별 청크 수:", dict(counts))
print()
print("예시 청크 꼬리표:", chunks[0].metadata)

분류별 청크 수: {'refund': 5, 'faq': 5, 'catalog': 7, 'shipping': 4, 'membership': 4}

예시 청크 꼬리표: {'source': 'winib_docs/02_refund.md', 'category': 'refund'}


**이 셀이 한 일**: 각 청크에 `category` 꼬리표를 달았습니다.
이제 청크마다 "어느 파일에서 왔고(source), 어떤 분류인지(category)"
정보가 모두 있습니다.

> 🌱 이 `category` 꼬리표가 "원두 질문엔 환불 문서만"
> 골라 검색하는 핵심 재료가 됩니다. 씨앗을 지금 심어둔 것입니다.

---
# 🎓 배운 것 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | 통째로의 한계 → 쪼개기 필요 | 불러오기 + 쪼개기 |
| 2️⃣ 기본 | `loader.load()` / `split_documents()` | 본문+꼬리표 |
| 3️⃣ 응용 | size·overlap 바꿔보기 | 적당한 크기 감각 |
| 4️⃣ 실습 | 위니브 5문서 → 청크 + 분류 꼬리표 | 5교시 검색 준비 |

**한 문장 요약**: 문서를 **불러와서** 검색하기 좋게 **잘게 쪼개고**,
각 조각에 출처·분류 **꼬리표**를 달아둔다.

**다음 시간**: 잘라둔 청크를 "글자가 아니라 **의미**로
검색"되게 만듭니다 — 임베딩과 벡터 데이터베이스.

---
## 🤔 생각해보기 

위니브 배송정책(`03_shipping.md`)에는 **표(table)** 가 들어 있습니다.

1. 표가 든 문서를 일반 문서와 똑같은 크기로 자르면 어떤 문제가
   생길까요?
2. 표가 안 잘리게 하려면 chunk_size를 어떻게 하면 좋을까요?
   그 부작용은요?
3. "표는 표대로, 글은 글대로" 다르게 처리하려면 어떤 아이디어가
   필요할까요? 

> ✍️ 메모:
> - 1번:
> - 2번:
> - 3번:
